# Problem 4: Cracking the Bible Code

---

The Bible code is a purported set of encoded words within a Hebrew text of the Torah that, according to proponents, has predicted significant historical events. The statistical likelihood of the Bible code predictions arising by chance has been thoroughly researched, and it is now widely considered to be statistically insignificant, as similar phenomena can be observed in any sufficiently lengthy text [1]. Although Bible codes have been postulated and studied for centuries, the subject was popularized in modern times by Michael Drosnin's book *The Bible Code* and the movie *The Omega Code*.

In 1994, a peer-reviewed academic journal published a paper reporting statistically significant codes in the Bible as a "challenging puzzle" [2]. It was pronounced "solved" in a subsequent 1999 paper published in the same journal [3].

In the original paper, the authors used **Equidistant Letter Sequences (ELS)** to identify words and names within the text. Using this technique, others have claimed to have correlated names, dates, and historical events, such as the assassination of John F. Kennedy or the September 11 attacks in New York.

Today, the Bible Code has been largely dismissed as statistically insignificant — simply an amusing byproduct of searching a large body of text. Read the excerpt of *The Baltimore Stockbroker and the Bible Code* from Jordan Ellenberg's *How Not to Be Wrong*, located in [`research/`](../research/ellenberg-2014-bible-code.pdf), for historical context. *Despite the underlying statistics, there are still two distinct camps of people when it comes to the Bible Code: those who believe and those who don't.*

<p><iframe title="YouTube video player" src="https://www.youtube.com/embed/Lk3VgQgxiqE" width="560" height="315" allowfullscreen="allowfullscreen" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share"></iframe></p>

---

## The ELS Algorithm

The ELS algorithm extracts every $k$-th character from a text to form a new string, then searches that string for meaningful words. Before applying ELS, the text must be **preprocessed**: remove all spaces and punctuation, and convert to lowercase. The resulting string is referred to as the *cleaned text*.

For a given stride $k$, the extracted string is:

$$\text{ELS}(k) = c_0,\ c_k,\ c_{2k},\ c_{3k},\ \ldots$$

where $c_i$ is the $i$-th character of the cleaned text.

### Example

Given the opening of Genesis 1:1 — *"In the beginning God created the heaven and the earth"* — the cleaned text is:

```
inthebeginninggodcreatedtheheavenandtheearth
```

Applying ELS with strides 2, 3, and 4:

``` 
stride 2 → iteeinngdraeteevnnteat     words found: it, tee, in, tea
stride 3 → ihennortthvatet            words found: hen, or, vat
stride 4 → ieindatenta                words found: in, date
```

Words are identified by scanning the ELS-extracted string for **substrings that match known words**. To avoid trivially short matches, only words of **4 or more characters** should be counted. Words are matched against the reference lists described below.

![Figure 1. ELS stride extraction and same-stride proximity phrase detection. Top: every 3rd character is selected from the cleaned text to form the extracted string. Bottom: two named words found close together in the same extracted string (stride 47) are recorded as a phrase co-occurrence.](assets/bible-code.png)

*Figure 1. ELS stride extraction (top) and same-stride proximity phrase detection (bottom).*

---

## Task

Write a program using the ELS algorithm to search for encoded words across three texts: the **King James Bible**, **War and Peace**, and **Moby Dick**. All texts are available in `data/`. Your goal is to determine whether the encoded words found in the Bible represent something genuinely prophetic, or whether they are simply a statistical artifact of searching any sufficiently large body of text.

### Reference Lists

Match your ELS-extracted strings against the following, all located in `data/`:

- `words_10k.txt` — the 10,000 most commonly used English words (minimum match length: 4 characters)
- `names_and_dates.txt` — a curated list of famous names and historical events drawn from the original Bible code literature, including the assassinations, historical events, and scientific figures that proponents claimed to have found encoded in the Torah

Feel free to supplement these with your own reference strings (e.g., specific dates, names, events) if you wish to search for something particular.

### Stride Range

Search strides from **2 to 1,000**. This keeps output files to a manageable size and ensures every extracted string is at least 1,100 characters long (for Moby Dick), which is sufficient to find meaningful words and phrase co-occurrences. The interesting results in the original Bible code research were overwhelmingly found at small strides where extracted strings are long and dense — searching beyond 1,000 yields diminishing returns.

### Finding Phrases

Individual words found by ELS are interesting, but the original Bible code claims were based on finding **pairs or groups of related words appearing close together** within the same ELS-extracted string. This is known as **same-stride proximity**: two words form a meaningful co-occurrence if they are found by the *same stride value* and appear *near each other* in the resulting extracted string.

The logic is as follows: if stride $k$ extracts a string and that string contains both `kennedy` and `assassin` close together, then both words were encoded at the same skip interval and in nearby positions in the original text — a potentially non-random clustering. If the same two words appear far apart, or are found at completely different strides, the association is much weaker.

Phrases are identified **on the fly** — within the same loop that finds words. Once the matches for a given stride are in memory, pairs are checked immediately before moving to the next stride. This avoids a second pass over the output files and keeps memory usage low, since only one stride's matches are held at a time.

> **Important:** Only search for co-occurrences using `names_and_dates.txt` — **not** the 10,000 word list. The 10k list produces so many matches per stride that the number of pairs grows quadratically and output files can reach gigabyte scale. Restricting phrase-finding to named entities keeps the output sparse and the results meaningful — coincidental pairings of common words like `that` and `with` are not interesting.

> **Note:** The proximity window of 25 is arbitrary — try adjusting it and observe how the number of co-occurrences changes. A very large window will produce many spurious "phrases"; a very small one may miss genuine clusters.

### Output Files

Write results to two separate files per text, stored in `output/biblecode/`:

```
output/biblecode/bible-words.txt
output/biblecode/bible-phrases.txt

output/biblecode/moby-dick-words.txt
output/biblecode/moby-dick-phrases.txt

output/biblecode/warandpeace-words.txt
output/biblecode/warandpeace-phrases.txt
```

**Words file** — one match per line:

```
stride,pos,word
2,18,date
2,441,kennedy
7,1204,lincoln
312,88431,hurricane
```

**Phrases file** — one co-occurrence group per line. After the stride, list every word in the proximity cluster as `position,word` pairs. There is no limit on how many words can form a phrase — any number of words found within the proximity window at the same stride are grouped together on a single line. The more words on a line, the more striking the co-occurrence:

```
stride,pos,word,pos,word,pos,word,pos,word
47,120,kennedy,134,assassin
312,880,lincoln,891,assassination,904,dallas
89,210,hitler,218,nazi,225,genocide,231,war
```

Keeping them separate makes each file easier to query and analyse independently — word matches and phrase co-occurrences answer different questions and will be processed differently in your notebook.

---

## Post to Slack

Post your most striking "prediction" in the `#showcase` Slack channel: the phrase with the most words, or the one that reads most like a prophecy. Use this format, from a line of your phrases file:

```
NAME:     <your name>
TEXT:     <Bible, Moby Dick or War and Peace>
STRIDE:   <stride>
PHRASE:   <pos,word,pos,word,...>
```

For example:

```
NAME:     Ada Lovelace
TEXT:     Moby Dick
STRIDE:   312
PHRASE:   880,lincoln,891,assassination,904,dallas
```

---

## References

[1] McKay, B., Bar-Natan, D., Bar-Hillel, M., & Kalai, G. (1999). Solving the Bible Code Puzzle. *Statistical Science*, 14(2), 150–173. https://doi.org/10.1214/ss/1009212243

[2] Witztum, D., Rips, E., & Rosenberg, Y. (1994). Equidistant Letter Sequences in the Book of Genesis. *Statistical Science*, 9(3), 429–438. https://doi.org/10.1214/ss/1177010393

[3] Kass, R.E. (1999). Introduction to "Solving the Bible Code Puzzle". *Statistical Science*, 14(2), 149. https://doi.org/10.1214/ss/1009212242

In [7]:
from pathlib import Path
from collections import defaultdict


# Preprocessing the text
def clean_text(text):
    return "".join(char.lower() for char in text if char.isalpha())


# Loading the text from a file
def load_text(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return clean_text(file.read())


# Loading reference words
def load_words(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        return {
            word
            for line in file
            if len(word := clean_text(line.strip())) >= 4
        }


# Grouping words based on their first character
def build_word_index(all_words):
    word_index = defaultdict(list)

    for word in all_words:
        word_index[word[0]].append(word)

    return word_index


# Finding all matching words and named entities
def find_words(els_text, word_index, names):
    entity_matches = []

    for pos, char in enumerate(els_text):

        # Check only words beginning with the current character
        for word in word_index.get(char, []):

            if els_text.startswith(word, pos):
                yield pos, word, word in names


# Finding phrase clusters
def find_phrases(entity_matches, window=25):
    phrases = []
    cluster = []

    for pos, word in entity_matches:

        if not cluster:
            cluster = [(pos, word)]

        elif pos - cluster[0][0] <= window:
            cluster.append((pos, word))

        else:
            if len(cluster) >= 2:
                phrases.append(cluster)

            cluster = [(pos, word)]

    if len(cluster) >= 2:
        phrases.append(cluster)

    return phrases


# Process one book
def process_book(text, book_name, word_index, names):

    output_dir = Path("../output/biblecode")
    output_dir.mkdir(parents=True, exist_ok=True)

    words_path = output_dir / f"{book_name}-words.txt"
    phrases_path = output_dir / f"{book_name}-phrases.txt"

    with open(words_path, "w", encoding="utf-8") as words_file, \
         open(phrases_path, "w", encoding="utf-8") as phrases_file:

        for stride in range(2, 1001):

            # Extracting ELS string
            els_text = text[::stride]

            entity_matches = []

            # Finding words and named entities together
            for pos, word, is_entity in find_words(els_text, word_index, names):

                # Writing all word matches
                words_file.write(f"{stride},{pos},{word}\n")

                # Storing only named entity matches
                if is_entity:
                    entity_matches.append((pos, word))

            # Find phrases for the current stride
            phrases = find_phrases(entity_matches, window=25)

            # Writing phrases immediately
            for phrase in phrases:
                values = [str(stride)]

                for pos, word in phrase:
                    values.extend([str(pos), word])

                phrases_file.write(",".join(values) + "\n")

    print(f"Completed processing {book_name}")



words = load_words("../data/words_10k.txt")
names = load_words("../data/names_and_dates.txt")

all_words = words | names

word_index = build_word_index(all_words)

books = {
    "bible": "../data/bible.txt",
    "warandpeace": "../data/war-and-peace.txt",
    "moby-dick": "../data/moby-dick.txt"
}

for book_name, file_path in books.items():
    text = load_text(file_path)
    process_book(text, book_name, word_index, names)

Completed processing bible
Completed processing warandpeace
Completed processing moby-dick
